# SAM 3D Objects on Kaggle

This notebook follows Meta's official SAM 3D Objects setup and inference flow. Before running it, enable **Internet** and add a Kaggle secret named `HF_TOKEN` containing a Hugging Face token that has been granted access to `facebook/sam-3d-objects`.

> **Staged 16 GB mode:** Meta recommends 32 GB VRAM on one GPU. Kaggle's two T4s do not pool memory, so this notebook keeps the official SAM 3D Objects checkpoints in host RAM and moves only the depth, sparse-structure, SLAT, or Gaussian-decoder stage to one T4 at a time. It exports a Gaussian PLY only. This is a memory-constrained execution path, not Meta's supported 32 GB configuration.


## Kaggle checklist

1. In Notebook options, turn on Internet and select a GPU session.
2. Request access to the gated `facebook/sam-3d-objects` Hugging Face model.
3. Add your approved Hugging Face token to Kaggle Secrets as `HF_TOKEN`.
4. Attach an image and a matching binary mask as a Kaggle Dataset, then set their paths in the input cell below.


In [ ]:
import subprocess
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
gpu_count = torch.cuda.device_count()
gpu_info = [
    {
        'index': index,
        'name': torch.cuda.get_device_name(index),
        'vram_gb': round(torch.cuda.get_device_properties(index).total_memory / 2**30, 1),
    }
    for index in range(gpu_count)
]
print(gpu_info)

max_vram_gb = max(item['vram_gb'] for item in gpu_info)
LOW_VRAM_MODE = max_vram_gb < 32
if LOW_VRAM_MODE:
    print('WARNING: running best-effort low-VRAM mode on one 16 GB GPU. GPU 1 is not pooled with GPU 0.')
    print('The runner will export Gaussian splats only and downsample sparse structure aggressively.')
else:
    print('32 GB+ GPU detected; low-VRAM limits can be relaxed in the input cell.')

print('GPU preflight passed.')
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'sam-3d-objects'
# Use Kaggle's supplied CUDA Python. Installing a separate Conda runtime can
# stall before any useful SAM 3D work begins.
PYTHON = Path(sys.executable)

if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/facebookresearch/sam-3d-objects.git', str(REPO_DIR)
    ], check=True)

print('Repository:', REPO_DIR)
print('Python:', PYTHON)


In [ ]:
# Inference runtime closure for Meta's released Gaussian pipeline. This deliberately
# excludes unrelated dev tools (Jupyter, Blender, SageMaker, point-cloud-utils),
# but includes every non-optional module imported by the released pipeline.
env = os.environ.copy()
env['PIP_NO_CACHE_DIR'] = '1'
env['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
env['MAX_JOBS'] = '2'  # keeps source builds within Kaggle host-memory limits
# Kaggle's prebuilt cumm/spconv wheels must not enter their editable-install
# JIT fallback: it expects C++ headers that wheels deliberately do not ship.
# These are the projects' documented switches for using the bundled CUDA binary.
env['CUMM_DISABLE_JIT'] = '1'
env['SPCONV_DISABLE_JIT'] = '1'
# Meta's released notebook sets this before importing sam3d_objects. The public
# package otherwise imports a non-distributed optional initialization module.
env['LIDRA_SKIP_INIT'] = 'true'
env['PIP_FIND_LINKS'] = 'https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.1_cu121.html'
CONSTRAINTS = WORKDIR / 'sam3d-constraints.txt'
CONSTRAINTS.write_text('numpy==1.26.4\n')

def pip_install(*packages, extra=()):
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--no-cache-dir', '--constraint', str(CONSTRAINTS), *extra, *packages], env=env, check=True)

def check_imports(code, label):
    subprocess.run([str(PYTHON), '-c', code], cwd=REPO_DIR, env=env, check=True)
    print(label)

# Pin the CUDA 12.1 stack expected by Kaolin and Meta's released SAM 3D setup.
pip_install('torch==2.5.1', 'torchvision==0.20.1', extra=('--index-url', 'https://download.pytorch.org/whl/cu121'))
# Keep NumPy 1.26.4 fixed across all installs. The selected SAM 3D Gaussian path
# does not instantiate the optional Kaolin mesh decoder.
pip_install(
    'numpy==1.26.4', 'astor==0.8.1', 'easydict', 'einops-exts', 'fvcore', 'huggingface-hub[cli]<1.0',
    'hydra-core==1.3.2', 'igraph==0.11.8', 'lightning==2.3.3', 'loguru==0.7.2',
    'matplotlib==3.9.2', 'omegaconf', 'open3d==0.19.0', 'optree==0.14.1',
    'opencv-python-headless==4.9.0.80', 'pillow', 'plyfile', 'pymeshfix==0.17.0',
    'pyvista==0.44.2', 'roma==1.5.1', 'safetensors', 'scipy', 'timm==0.9.16',
    # Kaggle's preinstalled Transformers 5 requires Hub >=1, conflicting with
    # Meta's documented Hub <1 requirement. This release supports Hub 0.23..<1.
    'transformers==4.46.3',
    'tqdm', 'trimesh', 'xatlas==0.0.9',
)
# Required even for Gaussian-only output: Meta's decoder imports the shared
# representations package, which imports FlexiCubes and Kaolin at module load.
# This exact cp312 / torch-2.5.1 / CUDA-12.1 wheel is present in NVIDIA's index.
pip_install('kaolin==0.17.0')
# These CUDA extensions are imported by the released pipeline. FlashAttention is
# deliberately omitted: on a T4 the pipeline uses PyTorch SDPA, so compiling it
# wastes time and disk without being used.
pip_install('spconv-cu121==2.3.8')
# Stop before the costly source builds if the NumPy or sparse-convolution ABI changes.
check_imports("import numpy; assert numpy.__version__ == '1.26.4', numpy.__version__; import spconv", 'NumPy/spconv ABI preflight passed.')
pip_install('git+https://github.com/facebookresearch/pytorch3d.git@75ebeeaea0908c5527e7b1e305fbc7681382db47')
check_imports('import pytorch3d', 'PyTorch3D preflight passed.')
pip_install('moderngl')
# This exact 0.0.2 release is only available from the MoGe-pinned source commit.
pip_install('git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900', extra=('--no-deps',))
# MoGe declares an unpinned NumPy dependency. Its inference imports use only the
# packages installed above, so prevent pip from replacing Kaolin-compatible NumPy.
pip_install('MoGe @ git+https://github.com/microsoft/MoGe.git@a8c37341bc0325ca99b9d57981cc3bb2bd3e255b', extra=('--no-deps',))
pip_install('gsplat @ git+https://github.com/nerfstudio-project/gsplat.git@2323de5905d5e90e035f792fe65bad0fedd413e7')
# The runner is executed from REPO_DIR, so Python can import sam3d_objects
# directly. Avoid an editable install, which would only add packaging work.
subprocess.run([str(REPO_DIR / 'patching' / 'hydra')], cwd=REPO_DIR, env=env, check=True)

# Fail before downloading SAM 3D checkpoints if any import reachable from the
# released PointMap/Gaussian configuration is missing. The selected GS decoder
# uses `attn_mode: swin`, so its optional vox2seq serialized-attention path is not used.
# Force the portable SDPA path before SAM 3D modules are imported.
env['ATTN_BACKEND'] = 'sdpa'
env['SPARSE_ATTN_BACKEND'] = 'sdpa'
check_imports("import importlib, numpy, torch; assert numpy.__version__ == '1.26.4', numpy.__version__; import astor, igraph, kaolin, matplotlib, open3d, optree, pymeshfix, pyvista, spconv, timm, transformers, pytorch3d, gsplat; assert transformers.__version__ == '4.46.3', transformers.__version__; modules=('moge.model.v1','sam3d_objects.config.utils','sam3d_objects.data.dataset.tdfy.img_and_mask_transforms','sam3d_objects.data.dataset.tdfy.img_processing','sam3d_objects.data.dataset.tdfy.preprocessor','sam3d_objects.model.backbone.dit.embedder.dino','sam3d_objects.model.backbone.dit.embedder.embedder_fuser','sam3d_objects.model.backbone.dit.embedder.pointmap','sam3d_objects.model.backbone.generator.classifier_free_guidance','sam3d_objects.model.backbone.generator.flow_matching.model','sam3d_objects.model.backbone.generator.shortcut.model','sam3d_objects.model.backbone.tdfy_dit.models.mm_latent','sam3d_objects.model.backbone.tdfy_dit.models.mot_sparse_structure_flow','sam3d_objects.model.backbone.tdfy_dit.models.sparse_structure_vae','sam3d_objects.model.backbone.tdfy_dit.models.structured_latent_flow','sam3d_objects.model.backbone.tdfy_dit.models.structured_latent_vae.decoder_gs','sam3d_objects.pipeline.depth_models.moge','sam3d_objects.pipeline.inference_pipeline_pointmap'); [importlib.import_module(module) for module in modules]; print(torch.__version__, torch.version.cuda, numpy.__version__)", 'Full released SAM 3D Gaussian runtime import preflight passed.')
print('Lean SAM 3D inference environment is ready.')


In [ ]:
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
assert hf_token, 'Add a Kaggle secret named HF_TOKEN after Hugging Face grants model access.'

env = os.environ.copy()
env['HF_TOKEN'] = hf_token
download_dir = REPO_DIR / 'checkpoints' / 'hf-download'
target_dir = REPO_DIR / 'checkpoints' / 'hf'

import shutil
if not target_dir.exists():
    free_gb = shutil.disk_usage(WORKDIR).free / 2**30
    print(f'Free disk before checkpoints: {free_gb:.1f} GB')
    assert free_gb >= 20, 'Less than 20 GB free: stop before the 13 GB checkpoint download.'
    from huggingface_hub import snapshot_download
    # Download only the released checkpoints/configs used by Gaussian-only staged
    # inference; mesh and encoder files are intentionally excluded.
    gaussian_files = [
        'checkpoints/pipeline.yaml',
        'checkpoints/ss_generator.ckpt', 'checkpoints/ss_generator.yaml',
        'checkpoints/ss_decoder.ckpt', 'checkpoints/ss_decoder.yaml',
        'checkpoints/slat_generator.ckpt', 'checkpoints/slat_generator.yaml',
        'checkpoints/slat_decoder_gs.ckpt', 'checkpoints/slat_decoder_gs.yaml',
    ]
    # local_dir resumes interrupted files, including after a quick-save restart.
    snapshot_download('facebook/sam-3d-objects', allow_patterns=gaussian_files, local_dir=str(download_dir), token=hf_token, max_workers=1)
    (download_dir / 'checkpoints').rename(target_dir)
else:
    print('Reusing existing checkpoints; no model download is needed.')
print('Checkpoints:', target_dir)


## Input image and mask

Set these to a color image and matching binary mask (white = object, black = background). In SketchScape, SAM 3 would normally produce this mask before SAM 3D Objects runs.


In [ ]:
# This default uses Meta's bundled example. Replace both paths with an attached
# Kaggle Dataset for your own SketchScape object and binary mask.
EXAMPLE_DIR = REPO_DIR / 'notebook' / 'images' / 'shutterstock_stylish_kidsroom_1640806567'
INPUT_IMAGE = str(EXAMPLE_DIR / 'image.png')
INPUT_MASK = str(EXAMPLE_DIR / '14.png')
OUTPUT_DIR = '/kaggle/working/sam3d-output'

# Low-VRAM controls. A single image has batch size 1 already. The staged runner
# is the memory control; these trade reconstruction detail and time for a safer
# first 16 GB T4 run. Increase only after a successful output.
DOWNSAMPLE_SS_DIST = 4 if LOW_VRAM_MODE else 1
MAX_INPUT_SIDE = 768 if LOW_VRAM_MODE else 1536
STAGE1_STEPS = 12 if LOW_VRAM_MODE else 25
STAGE2_STEPS = 12 if LOW_VRAM_MODE else 25
SEED = 42

assert Path(INPUT_IMAGE).is_file(), f'Missing image: {INPUT_IMAGE}'
assert Path(INPUT_MASK).is_file(), f'Missing mask: {INPUT_MASK}'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# Run official SAM 3D Objects in explicit stages. The public pipeline loads all
# checkpoints onto one GPU; this runner retains the official checkpoints but keeps
# inactive stages on CPU. It does not claim to combine the two T4 memories.
runner = REPO_DIR / 'run_kaggle_inference.py'
runner.write_text(f'''
import gc
import os
from pathlib import Path

import numpy as np
import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf
from PIL import Image

# T4 uses the torch SDPA code path, so no FlashAttention build is required.
os.environ['ATTN_BACKEND'] = 'sdpa'
os.environ['SPARSE_ATTN_BACKEND'] = 'sdpa'
from sam3d_objects.pipeline.inference_pipeline import InferencePipeline

repo = Path({str(REPO_DIR)!r})
image_path = Path({INPUT_IMAGE!r})
mask_path = Path({INPUT_MASK!r})
output_dir = Path({OUTPUT_DIR!r})
output_dir.mkdir(parents=True, exist_ok=True)
gpu = torch.device('cuda:0')
cpu = torch.device('cpu')

def clear_cuda(label):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    used = torch.cuda.memory_allocated() / 2**30
    reserved = torch.cuda.memory_reserved() / 2**30
    print(f'{{label}}: CUDA allocated={{used:.2f}} GB, reserved={{reserved:.2f}} GB')

def move_models(names, device):
    for name in names:
        model = pipeline.models[name]
        if model is not None:
            model.to(device)
    conditioner_names = {{
        'ss_generator': 'ss_condition_embedder',
        'slat_generator': 'slat_condition_embedder',
    }}
    for name in names:
        conditioner = pipeline.condition_embedders.get(conditioner_names.get(name, ''))
        if conditioner is not None:
            conditioner.to(device)

def move_depth(device):
    pipeline.depth_model.model.to(device)
    pipeline.depth_model.device = device

# The output is Gaussian-only. Do not construct the optional mesh decoder (and
# therefore do not require Kaolin or its mesh checkpoint) for this execution.
def skip_mesh_decoder(self, *args, **kwargs):
    return None

InferencePipeline.init_slat_decoder_mesh = skip_mesh_decoder

# Load every official checkpoint into host RAM first. Setting both devices before
# Hydra instantiates the graph is essential: the depth wrapper defaults to CUDA.
base_config = repo / 'checkpoints' / 'hf' / 'pipeline.yaml'
config = OmegaConf.load(base_config)
config.workspace_dir = str(base_config.parent)
config.device = 'cpu'
config.depth_model.device = 'cpu'
config.compile_model = False
config.decode_formats = ['gaussian']
config.slat_decoder_gs_4_config_path = None
config.slat_decoder_gs_4_ckpt_path = None
config.slat_decoder_mesh_config_path = None
config.slat_decoder_mesh_ckpt_path = None
config.downsample_ss_dist = int({DOWNSAMPLE_SS_DIST!r})
pipeline = instantiate(config)
pipeline.device = gpu
print('Loaded official SAM 3D checkpoints on CPU; downsample_ss_dist =', config.downsample_ss_dist)

image = np.asarray(Image.open(image_path).convert('RGB'))
mask = np.asarray(Image.open(mask_path).convert('L')) > 127
assert image.shape[:2] == mask.shape[:2], 'Image and mask dimensions must match.'
max_input_side = int({MAX_INPUT_SIDE!r})
height, width = image.shape[:2]
if max(height, width) > max_input_side:
    scale = max_input_side / max(height, width)
    size = (round(width * scale), round(height * scale))
    image = np.asarray(Image.fromarray(image).resize(size, Image.Resampling.LANCZOS))
    mask = np.asarray(Image.fromarray(mask.astype(np.uint8) * 255).resize(size, Image.Resampling.NEAREST)) > 127
    print('Resized input to %dx%d.' % (image.shape[1], image.shape[0]))
rgba = np.concatenate([image, (mask.astype(np.uint8) * 255)[..., None]], axis=-1)

# Stage 0: MoGe point map. Its weights are released before stage 1 begins.
move_depth(gpu)
clear_cuda('depth loaded')
with torch.no_grad():
    pointmap = pipeline.compute_pointmap(rgba)['pointmap']
move_depth(cpu)
clear_cuda('depth offloaded')

# Stage 1: sparse structure. The matching conditioner, generator, and decoder are
# concurrently on GPU only for this call.
ss_input = pipeline.preprocess_image(rgba, pipeline.ss_preprocessor, pointmap=pointmap)
del pointmap
move_models(('ss_generator', 'ss_decoder'), gpu)
clear_cuda('stage 1 loaded')
torch.manual_seed(int({SEED!r}))
ss_return = pipeline.sample_sparse_structure(ss_input, inference_steps=int({STAGE1_STEPS!r}))
ss_return.update(pipeline.pose_decoder(
    ss_return,
    scene_scale=ss_input.get('pointmap_scale'),
    scene_shift=ss_input.get('pointmap_shift'),
))
ss_return['scale'] = ss_return['scale'] * ss_return['downsample_factor']
coords = ss_return['coords']
move_models(('ss_generator', 'ss_decoder'), cpu)
del ss_input, ss_return
clear_cuda('stage 1 offloaded')

# Stage 2: SLAT. Coordinates remain small enough to keep on GPU; the large stage-1
# weights are already back in RAM.
slat_input = pipeline.preprocess_image(rgba, pipeline.slat_preprocessor)
move_models(('slat_generator',), gpu)
clear_cuda('stage 2 loaded')
slat = pipeline.sample_slat(slat_input, coords, inference_steps=int({STAGE2_STEPS!r}))
move_models(('slat_generator',), cpu)
del slat_input, coords
clear_cuda('stage 2 offloaded')

# Stage 3: decode only the requested Gaussian representation. No mesh conversion,
# texture baking, rendering, or layout optimization is invoked.
move_models(('slat_decoder_gs',), gpu)
clear_cuda('Gaussian decoder loaded')
outputs = pipeline.decode_slat(slat, formats=['gaussian'])
gaussian = outputs['gaussian'][0]
gaussian.save_ply(output_dir / 'reconstruction.ply')
print('Saved:', output_dir / 'reconstruction.ply')
del outputs, gaussian, slat
move_models(('slat_decoder_gs',), cpu)
clear_cuda('complete')
''')

runner_env = os.environ.copy()
runner_env['CUDA_VISIBLE_DEVICES'] = '0'  # stage offloading uses one T4 only
runner_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
runner_env['ATTN_BACKEND'] = 'sdpa'
runner_env['SPARSE_ATTN_BACKEND'] = 'sdpa'
runner_env['CUMM_DISABLE_JIT'] = '1'
runner_env['SPCONV_DISABLE_JIT'] = '1'
runner_env['LIDRA_SKIP_INIT'] = 'true'
subprocess.run([str(PYTHON), str(runner)], cwd=REPO_DIR, env=runner_env, check=True)

# Keep the PLY only. Removing the recreatable runtime and checkpoints prevents
# Kaggle from failing while saving this notebook's output version.
for generated_path in (target_dir,):
    if generated_path.exists():
        shutil.rmtree(generated_path)
print('Removed temporary runtime and checkpoints; output remains in', OUTPUT_DIR)


## Hand-off to Unity

The official quick start exports `reconstruction.ply` (Gaussian splat), not a `.glb`. Preserve that output for validation, then add a dedicated mesh/GLB conversion step before writing the resulting asset URL into SketchScape scene JSON. Do not label a PLY as a GLB or send it directly to the current Unity GLB loader.
